In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,                                # 교차검증시 비율 유지
    RandomizedSearchCV,                             # 하이퍼파라미터 조합 무작위 고르기
    cross_val_score                                 # 교차검증 점수를 계산
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,                         # 혼동행렬 그리기
    RocCurveDisplay
)
from sklearn.linear_model import LogisticRegression # 로지스틱 회귀
from sklearn.ensemble import RandomForestClassifier # 랜덤 포레스트
from xgboost import XGBClassifier                   # 고성능 부스팅 모델
from lightgbm import LGBMClassifier                 # 다른 고성능 부스팅 모델
import optuna                                       # 하이퍼파라미터 자동 튜닝 도구

In [3]:
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

In [4]:
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target_original'] = data.target

df['target'] = 1 - df['target_original']
df['target_name'] = df['target'].map({
    0: 'benign',
    1: 'malignant'
})

In [5]:
X = df.drop(columns=['target_original', 'target', 'target_name'])
y = df['target']
X.shape, y.shape

((569, 30), (569,))

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print('\ntrain의 target 비율:')
print(y_train.value_counts(normalize=True).round(3))
print('\ntest의 target 비율:')
print(y_test.value_counts(normalize=True).round(3))


(455, 30) (114, 30)

train의 target 비율:
target
0    0.626
1    0.374
Name: proportion, dtype: float64

test의 target 비율:
target
0    0.632
1    0.368
Name: proportion, dtype: float64


In [8]:
def evaluate_classification_model(model_name, y_true, y_pred, y_proba):
    """
    분류모델의 성능을 같은기준으로 평가하는 함수
    0 = benign, 양성
    1 = malignant, 악성

    y_proba : malignant일 확률
    """
    accuracy = accuracy_score(y_true, y_pred)
    precision_malignant = precision_score(y_true, y_pred, pos_label=1)
    recall_malignant = f1_score(y_true, y_pred, pos_label=1)
    f1_malignant = f1_score(y_true, y_pred, pos_label=1)
    roc_auc = roc_auc_score(y_true, y_proba)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    result = {
        'model_name': model_name,
        'accuracy': accuracy,
        'precision_malignant': precision_malignant,
        'recall_malignant': recall_malignant,
        'f1_malignant': f1_malignant,
        'roc_auc': roc_auc,
        'TN': tn,
        'FP': fp,
        'FN': fn,
        'TP': tp
    }

    return result

In [9]:
# 기본 베이스 모델
xgb_base_model = XGBClassifier(
    n_estimators=100,       # 트리 100개
    max_depth=3,
    learning_rate=0.1,
    random_state=42,        # 재현성 고정
    eval_metric='logloss'   # 학습 중 사용할 평가 기준
)

In [10]:
xgb_base_model.fit(X_train, y_train)

xgb_base_pred = xgb_base_model.predict(X_test)

xgb_base_proba = xgb_base_model.predict_proba(X_test)[:, 1]

In [ ]:
# 공통 평가 함수 성능 정리
xgb_base_result = evaluate_classification_model(
    model_name='XGBoost Base',
    y_true=y_test,
    y_pred=xgb_base_pred,
    y_proba=xgb_base_proba
)

xgb_base_result

{'model_name': 'XGBoost Base',
 'accuracy': 0.9649122807017544,
 'precision_malignant': 1.0,
 'recall_malignant': 0.95,
 'f1_malignant': 0.95,
 'roc_auc': 0.9966931216931217,
 'TN': np.int64(72),
 'FP': np.int64(0),
 'FN': np.int64(4),
 'TP': np.int64(38)}

In [ ]:
# 더 자세한 평가 결과